In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

print("All files loaded successfully.")

All files loaded successfully.


In [3]:
print("orders shape:", orders.shape)
print("customers shape:", customers.shape)
print()
orders.info()

orders shape: (99441, 8)
customers shape: (99441, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [4]:
orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [5]:
print("order_status value counts:")
print(orders["order_status"].value_counts())
print()

print("unique order_id:", orders["order_id"].nunique())
print("total rows in orders:", len(orders))
print()

print("unique customer_id:", customers["customer_id"].nunique())
print("unique customer_unique_id:", customers["customer_unique_id"].nunique())

order_status value counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

unique order_id: 99441
total rows in orders: 99441

unique customer_id: 99441
unique customer_unique_id: 96096


### Key finding: customer_id vs customer_unique_id

`customer_id` is unique per row (99,441 unique = 99,441 rows), but `customer_unique_id` has
only 96,096 unique values. This means `customer_id` is generated fresh per order, while
`customer_unique_id` identifies the actual person. 

**Rule for this project: always use `customer_unique_id` for any repeat-customer or
repurchase-rate logic. Never use `customer_id` for that purpose.**

In [6]:
n_before = len(orders)

orders_delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_estimated_delivery_date"].notna())
].copy()

n_after = len(orders_delivered)

print(f"Orders before filtering: {n_before}")
print(f"Orders after filtering to delivered + valid dates: {n_after}")
print(f"Orders dropped: {n_before - n_after}")

Orders before filtering: 99441
Orders after filtering to delivered + valid dates: 96470
Orders dropped: 2971


In [7]:
orders_delivered["delivery_delay_days"] = (
    orders_delivered["order_delivered_customer_date"] - orders_delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders_delivered["is_late"] = orders_delivered["delivery_delay_days"] > 0

print(orders_delivered["delivery_delay_days"].describe())
print()
print("% of orders delivered late:", round(orders_delivered["is_late"].mean() * 100, 1))

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64

% of orders delivered late: 8.1
